# 03 Temporal Reconstruction

**Project:** Black Hills Mining Landscape Digital Twin Phase I
**Territory:** He Sapa (Black Hills) Unceded Lakota Territory
**Data Sources:** USGS MRDS, Claude API (structured extraction)

## Purpose

When did mining begin in He Sapa? When did each commodity peak?
Which mines operated for decades, and which were short-lived rushes?

This notebook reconstructs the temporal arc of mining activity using:

1. **MRDS date fields** where available and reliable
2. **Claude API extraction** structured records pulled from USGS
   Mineral Resources bulletins and professional papers, with human
   review before gazetteer entry

## AI Extraction: Design Principles

The Claude API extraction pipeline follows three rules:

1. **AI extracts, humans verify.** No record enters the gazetteer
   without human review. The `human_verified` field tracks this.
2. **Uncertainty is explicit.** If the source says "ca. 1890" or
   "active in the early 1900s," the extracted record carries a
   `date_certainty: low` field, not a false precision.
3. **Provenance travels with data.** Every AI-extracted record
   carries `ai_generated: True`, the source document title,
   and the extraction date.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import json
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load Anthropic API key
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    print("WARNING: ANTHROPIC_API_KEY not found in .env")
    print("AI extraction will be skipped. MRDS dates only.")
    AI_AVAILABLE = False
else:
    import anthropic
    AI_AVAILABLE = True
    print("Anthropic API key loaded.")

from src.constants import (
    ERA_BINS, COMMODITY_COLORS, OUTPUTS_DIR, FIGURES_DIR,
    TREATY_PROVENANCE,
)
from src.sovereignty import print_data_acknowledgment, generate_citations, build_record_provenance

warnings.filterwarnings("ignore")
%matplotlib inline

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

gazetteer_path = OUTPUTS_DIR/"he_sapa_mine_gazetteer.geojson"
assert gazetteer_path.exists(), "Run notebook 01 first."
mines = gpd.read_file(gazetteer_path)
print(f"Mines loaded: {len(mines):,}")

In [ ]:
# Print data sovereignty ackhowledgement at the top of every notebook
print_data_acknowledgment(source_keys=["mrds", "claude_api", "usgs_bulletins"])

## MRDS Date Analysis

In [ ]:
# Parse active dates from MRDS
# MRDS stores dates inconsistently as free text, ranges, blanks
date_cols = [c for c in mines.columns if "date" in c.lower() or "yr" in c.lower() or "actv" in c.lower()]
print(f"Date-related columns in MRDS: {date_cols}")

def parse_year(val):
    if pd.isna(val) or str(val).strip() in ("", "0", "None"):
        return None
    s = str(val).strip()
    import re
    years = re.findall(r"\b(1[6-9]\d{2}|20[0-2]\d)\b", s)
    if years:
        return int(years[0])
    return None

if "actv_dt" in mines.columns:
    mines["start_year"] = mines["actv_dt"].apply(parse_year)
elif "prod_date" in mines.columns:
    mines["start_year"] = mines["prod_date"].apply(parse_year)
else:
    mines["start_year"] = None

n_dated = mines["start_year"].notna().sum()
print(f"\nRecords with parseable start year: {n_dated:,} of {len(mines):,}")
print(f"  ({n_dated/len(mines)*100:.1f}% of records have date information)")
print()
print("Date coverage gap note:")
print("  Many MRDS records lack dates. The AI extraction pipeline (below)")
print("  supplements these from USGS primary source documents.")

In [ ]:
# Assign era bins to dated records
def assign_era(year):
    if pd.isna(year):
        return "Unknown"
    for era_name, (start, end) in ERA_BINS.items():
        if (start is None or year >= start) and (end is None or year <= end):
            return era_name
    return "Unknown"

mines["era"] = mines["start_year"].apply(assign_era)

print("MINING ACTIVITY BY ERA")
era_counts = mines["era"].value_counts()
for era in list(ERA_BINS.keys()) + ["Unknown"]:
    n = era_counts.get(era, 0)
    bar = chr(9608) * int(n / max(era_counts) * 25) if max(era_counts) > 0 else ""
    print(f"  {era:<35}: {n:>4}  {bar}")

## Claude API Extraction Pipeline

In [ ]:
# Example extraction from a USGS bulletin text passage
# In production: feed OCR'd text from USGS IMAP 2445 quadrangle reports

EXAMPLE_BULLETIN_TEXT = """
The Homestake Mine at Lead, South Dakota, has been the largest gold
producer in the Western Hemisphere. Gold was discovered in 1876 by
Moses and Fred Manuel. The mine operated continuously from 1877 until
2002, producing over 40 million troy ounces of gold. Primary commodity
was gold (Au), with minor silver. The mine is located at approximately
44.35N, 103.77W in Lawrence County. Owner: Homestake Mining Company,
later Barrick Gold Corporation. The mine is adjacent to Whitewood Creek,
a tributary of the Belle Fourche River.
"""

EXTRACTION_PROMPT = """Extract structured mine information from this USGS bulletin text.
Return ONLY valid JSON with these fields:
{
  "mine_name": "string",
  "aliases": ["list of alternate names"],
  "latitude": number or null,
  "longitude": number or null,
  "county": "string",
  "state": "SD",
  "primary_commodity": "string",
  "secondary_commodities": ["list"],
  "active_start": number or null,
  "active_end": number or null,
  "date_certainty": "high|medium|low",
  "operator": "string or null",
  "adjacent_waterways": ["list of stream names"],
  "source_quote": "brief verbatim phrase supporting the dates"
}

Text:
""" + EXAMPLE_BULLETIN_TEXT

if AI_AVAILABLE:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role":"user","content": EXTRACTION_PROMPT}]
    )
    raw = response.content[0].text.strip()
    try:
        extracted = json.loads(raw)
        print("AI EXTRACTION RESULT:")
        print(json.dumps(extracted, indent=2))
    except json.JSONDecodeError:
        print("Raw response (JSON parse failed):")
        print(raw)
else:
    print("AI_AVAILABLE=False: showing expected output format:")
    extracted = {
        "mine_name": "Homestake Mine",
        "aliases": ["Homestake Gold Mine"],
        "latitude": 44.35, "longitude": -103.77,
        "county": "Lawrence", "state": "SD",
        "primary_commodity": "gold",
        "secondary_commodities": ["silver"],
        "active_start": 1877, "active_end": 2002,
        "date_certainty": "high",
        "operator": "Homestake Mining Company/Barrick Gold Corporation",
        "adjacent_waterways": ["Whitewood Creek", "Belle Fourche River"],
        "source_quote": "operated continuously from 1877 until 2002"
    }
    print(json.dumps(extracted, indent=2))

In [ ]:
# Build a gazetteer-ready record from the extraction
# This would be reviewed by a human before final entry

if "extracted" in dir() and extracted:
    prov = build_record_provenance(
        source_key="claude_api",
        ai_generated=True,
        human_verified=False,
        notes="Extracted from USGS bulletin — AWAITING HUMAN REVIEW",
    )
    record = {**extracted, **prov}
    print("GAZETTEER RECORD (pending human review):")
    print(f"  mine_name      : {record['mine_name']}")
    print(f"  active_start   : {record['active_start']}")
    print(f"  active_end     : {record['active_end']}")
    print(f"  ai_generated   : {record['ai_generated']}")
    print(f"  human_verified : {record['human_verified']}")
    print(f"  treaty_territory: {record['treaty_territory']}")
    print()
    print("STATUS: This record is in the review queue.")
    print("Set human_verified=True after confirming against source document.")

---
## 3. Temporal Visualization

In [ ]:
# Timeline of mining activity by commodity
dated = mines[mines["start_year"].notna()].copy()

if len(dated) > 10:
    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

    # Top: mine count by decade
    ax = axes[0]
    dated["decade"] = (dated["start_year"] // 10 * 10).astype(int)
    decade_counts = dated.groupby("decade").size()
    ax.bar(decade_counts.index, decade_counts.values, width=8,
           color="#C0392B", alpha=0.8)
    ax.set_ylabel("New mines opened per decade", fontsize=10)
    ax.set_title("He Sapa Mining Activity Temporal Reconstruction\n"
                 "MRDS records with parseable start dates",
                 fontsize=11, fontweight="bold")
    ax.axvline(1876, color="#2C3E50", linewidth=2, linestyle="--",
               alpha=0.8, label="Custer expedition/Gold Rush (1876)")
    ax.legend(fontsize=9)
    despine(ax)

    # Bottom: by commodity
    ax = axes[1]
    for group, color in COMMODITY_COLORS.items():
        subset = dated[dated["commodity_group"] == group] if "commodity_group" in dated.columns else pd.DataFrame()
        if subset.empty:
            continue
        decade_g = subset.groupby("decade").size()
        ax.bar(decade_g.index, decade_g.values, width=8,
               color=color, alpha=0.7, label=group)
    ax.set_xlabel("Decade", fontsize=10)
    ax.set_ylabel("Mine openings", fontsize=10)
    ax.set_title("By Commodity", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, ncol=3)
    despine(ax)

    plt.tight_layout()
    fig.savefig(FIGURES_DIR/"03_temporal_reconstruction.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: figures/03_temporal_reconstruction.png")
else:
    print(f"Only {len(dated)} dated records insufficient for decade chart.")
    print("AI extraction pipeline will supplement MRDS dates from primary sources.")

## Exports

In [ ]:
# Save mines with era assignments
mines_temporal = mines.copy()
if "era" in mines.columns:
    mines_temporal["era"] = mines["era"]

mines_temporal.to_file(
    OUTPUTS_DIR/"he_sapa_mines_temporal.geojson", driver="GeoJSON"
)
print("Exported: outputs/he_sapa_mines_temporal.geojson")

# Save AI extraction review queue template
review_queue = pd.DataFrame(columns=[
    "mine_name", "latitude", "longitude", "active_start", "active_end",
    "date_certainty", "primary_commodity", "operator", "adjacent_waterways",
    "source_document", "source_quote", "ai_generated", "human_verified",
    "reviewer", "review_date", "review_notes",
])
review_queue.to_csv(OUTPUTS_DIR / "ai_extraction_review_queue.csv", index=False)
print("Exported: outputs/ai_extraction_review_queue.csv")
print("  (Add AI-extracted records here for human review before gazetteer entry)")

In [ ]:
print(generate_citations(["mrds", "claude_api", "usgs_bulletins"]))